# MI Event Flow Agent 실행 데모

이 노트북은 `skmip-mi-agent` 프로젝트의 `event_flow_agent`가 `/invoke` API를 통해 어떻게 동작하는지 단계별로 확인하기 위한 데모 자료입니다.

전체 흐름은 다음과 같습니다.

1. 원천 record 조회
2. record별 LLM 분석
3. event 단위 추출
4. `tmm_mieventm` 테이블 저장
5. 저장된 event 재조회
6. 최종 요약 생성

> 사전 조건: `skmip_mcp_server`, `skmip-mi-agent`, 로컬 MySQL Docker DB가 실행 중이어야 합니다. 가상환경은 skmip-mi-agent에서 잡고 있는 가상환경으로 맞춰 주세요.


## 1. 실행 환경 확인

아래 값은 현재 데모 환경 기준입니다.

- Agent API: `http://localhost:3002/invoke`
- DB Host: `127.0.0.1`
- DB Port: `13306`
- DB Name: `midb`
- Target Table: `tmm_mieventm`


In [3]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(PROJECT_ROOT)

from apps.common.configs.settings import get_settings

cfg = get_settings()

AGENT_INVOKE_URL = "http://localhost:3002/invoke"

DB_HOST = cfg.METADATA_DB_HOST
DB_PORT = cfg.METADATA_DB_PORT
DB_NAME = cfg.METADATA_DB_NAME
DB_USER = cfg.METADATA_DB_USERNAME
DB_PASSWORD = cfg.METADATA_DB_PASSWORD

print("Agent URL:", AGENT_INVOKE_URL)
print("DB:", f"{DB_HOST}:{DB_PORT}/{DB_NAME}")

Agent URL: http://localhost:3002/invoke
DB: 127.0.0.1:13306/midb


## 2. API 요청 Body 준비

`limit`은 원천 record 조회 건수입니다. 현재 데모에서는 5건을 기준으로 실행합니다.


In [4]:
request_body = {
    "input": {
        "limit": 5
    },
    "config": {},
    "kwargs": {}
}

request_body


{'input': {'limit': 5}, 'config': {}, 'kwargs': {}}

## 3. 실행 전 DB 상태 확인

API 실행 전 `tmm_mieventm`에 몇 건이 적재되어 있는지 확인합니다.

> 이 단계는 DB 접속 비밀번호가 필요합니다. 비밀번호를 입력하지 않으면 건너뛰어도 됩니다.


In [5]:
import pandas as pd
from sqlalchemy import create_engine, text

def create_db_engine():
    return create_engine(
        f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
        pool_pre_ping=True,
    )

def get_event_count():
    engine = create_db_engine()
    with engine.connect() as conn:
        return conn.execute(text("select count(*) from tmm_mieventm")).scalar_one()

try:
    before_count = get_event_count()
    print("실행 전 tmm_mieventm 건수:", before_count)
except Exception as exc:
    before_count = None
    print("DB 사전 확인을 건너뜁니다:", exc)

실행 전 tmm_mieventm 건수: 11


## 4. `event_flow_agent` invoke API 실행

아래 셀은 실제 `/invoke` API를 호출합니다. 실행 중 LLM 호출이 포함되어 있어 시간이 걸릴 수 있습니다.


In [6]:
import requests
import json

headers = {
    "authorization": cfg.api_key,
    "Content-Type": "application/json",
    "aip-user": "test-user",
}

response = requests.post(
    AGENT_INVOKE_URL,
    json=request_body,
    headers=headers,
    timeout=300,
)

print("HTTP Status:", response.status_code)
print(response.text[:1000])

response.raise_for_status()
result = response.json()
result.keys()

HTTP Status: 200
{"output":{"summary":"전체 요약:  \n2026년 4월 15일, 글로벌 LNG 시장은 환율 및 금리 정책 불확실성, 미국 경제 불확실성, 지정학적 긴장 완화, 그리고 물류 공급망 변화 등 다양한 요인으로 변동성이 확대되고 있다. 미국과 유로존의 정책 변화, 유럽의 공급망 다변화, 호르무즈 해협의 물류 차질 등이 가격과 수요에 직접적인 영향을 미치고 있다. 지정학적 리스크 완화와 공급망 불확실성이 동시에 존재하며, 시장 심리 변화가 단기 가격 변동성에 중요한 역할을 하고 있다.\n\n핵심 키워드:  \n금리 및 환율 정책, 지정학적 리스크, 공급망/물류 변화\n\n중요한 이벤트:  \n1. 미·이란 평화협상 재개 및 휴전 연장 가능성으로 호르무즈 해협 긴장 완화, LNG 공급 리스크 감소  \n2. 호르무즈 해협 통항량 감소로 인한 LNG 공급 차질과 물류 불확실성, 단기 가격 변동성 확대","llm_result":"전체 요약:  \n2026년 4월 15일, 글로벌 LNG 시장은 환율 및 금리 정책 불확실성, 미국 경제 불확실성, 지정학적 긴장 완화, 그리고 물류 공급망 변화 등 다양한 요인으로 변동성이 확대되고 있다. 미국과 유로존의 정책 변화, 유럽의 공급망 다변화, 호르무즈 해협의 물류 차질 등이 가격과 수요에 직접적인 영향을 미치고 있다. 지정학적 리스크 완화와 공급망 불확실성이 동시에 존재하며, 시장 심리 변화가 단기 가격 변동성에 중요한 역할을 하고 있다.\n\n핵심 키워드:  \n금리 및 환율 정책, 지정학적 리스크, 공급망/물류 변화\n\n중요한 이벤트:  \n1. 미·이란 평화협상 재개 및 휴전 연장 가능성으로 호르무즈 해협 긴장 완화, LNG 공급 리스크 감소  \n2. 호르무즈 해협 통항량 감소로 인한 LNG 공급 차질과 물류 불확실성, 단기 가격 변동성 확대","rdb_result":[{"mi_event_id":60,"first_created":"2026-04-28T05:57:36","crt_

dict_keys(['output', 'metadata'])

## 5. 최종 요약 결과 확인

LLM이 DB에 저장된 event 목록을 바탕으로 생성한 최종 요약입니다.


In [7]:
output = result["output"]

print(output.get("summary", ""))


전체 요약:  
2026년 4월 15일, 글로벌 LNG 시장은 환율 및 금리 정책 불확실성, 미국 경제 불확실성, 지정학적 긴장 완화, 그리고 물류 공급망 변화 등 다양한 요인으로 변동성이 확대되고 있다. 미국과 유로존의 정책 변화, 유럽의 공급망 다변화, 호르무즈 해협의 물류 차질 등이 가격과 수요에 직접적인 영향을 미치고 있다. 지정학적 리스크 완화와 공급망 불확실성이 동시에 존재하며, 시장 심리 변화가 단기 가격 변동성에 중요한 역할을 하고 있다.

핵심 키워드:  
금리 및 환율 정책, 지정학적 리스크, 공급망/물류 변화

중요한 이벤트:  
1. 미·이란 평화협상 재개 및 휴전 연장 가능성으로 호르무즈 해협 긴장 완화, LNG 공급 리스크 감소  
2. 호르무즈 해협 통항량 감소로 인한 LNG 공급 차질과 물류 불확실성, 단기 가격 변동성 확대


## 6. 실행된 노드 확인

`event_flow_agent`가 어떤 순서로 실행되었는지 확인합니다.


In [8]:
executed_nodes = output.get("executed_nodes", [])
pd.DataFrame({"step": range(1, len(executed_nodes) + 1), "node": executed_nodes})


,step,node
0,1,read_record_source
1,2,analyze_records_with_llm
2,3,extract_events_with_llm
3,4,save_events
4,5,read_rdb
5,6,analyze_with_llm


## 7. 단계별 처리 건수 확인

원천 record 조회, record 분석, event 추출, DB 저장 결과를 요약해서 봅니다.


In [9]:
summary_metrics = {
    "source_records_count": output.get("source_records_count"),
    "record_analyses_count": output.get("record_analyses_count"),
    "extracted_events_count": output.get("extracted_events_count"),
    "saved_event_ids": output.get("saved_event_ids"),
    "errors": output.get("errors"),
}

summary_metrics


{'source_records_count': 5,
 'record_analyses_count': 5,
 'extracted_events_count': 6,
 'saved_event_ids': [55, 56, 57, 58, 59, 60],
 'errors': []}

## 8. 저장 후 재조회된 event 목록

`read_rdb` 노드에서 DB를 다시 조회한 결과입니다. 이 값이 최종 요약 LLM의 입력으로 사용됩니다.


In [10]:
events_df = pd.DataFrame(output.get("rdb_result", []))

display_columns = [
    "mi_event_id",
    "first_created",
    "event_title_ko",
    "one_line_summary",
    "level_1_code",
    "level_2_code",
    "occurred_at",
]

existing_columns = [col for col in display_columns if col in events_df.columns]
events_df[existing_columns] if not events_df.empty else events_df


,mi_event_id,first_created,event_title_ko,one_line_summary,level_1_code,level_2_code,occurred_at
0,60,2026-04-28T05:57:36,달러 약세 및 금리 정책 불확실성에 따른 LNG 거래 비용 변동,달러 약세와 금리 정책 불확실성이 LNG 거래 비용 및 환율 변동에 영향을 미침.,macroeconomy_policy,interest_rate_fx_policy_change,2026-04-15T19:33:10
1,59,2026-04-28T05:57:36,미국 경제 불확실성 확대와 LNG 수요 영향,미국 내 경제 불확실성 확대가 LNG 수요 약화 및 지역별 수요 경쟁에 영향을 미침.,macroeconomy_policy,country_economic_condition_change,2026-04-15T20:20:50
2,58,2026-04-28T05:57:36,유럽 구매자들의 캐나다 LNG 공급망 다변화 및 파나마운하 경유 논의,유럽 구매자들이 캐나다 LNG를 파나마운하 경유로 운송하는 방안을 논의하며 공급망 ...,logistics_shipping,cargo_flow_change_port_canal_disruption,2026-04-15T19:12:50
3,57,2026-04-28T05:57:36,미·이란 평화협상 재개 및 휴전 연장 가능성에 따른 지정학적 긴장 완화,미·이란 평화협상 재개 및 휴전 연장 가능성이 호르무즈 해협 긴장 완화와 LNG 공...,geopolitical_risk,geopolitical_tension_instability,2026-04-15T19:33:10
4,56,2026-04-28T05:57:36,호르무즈 해협 통항량 감소에 따른 LNG 공급 차질 및 물류 불확실성,호르무즈 해협 통항량 감소가 LNG 및 원유 운송에 차질을 초래하며 글로벌 공급망 ...,logistics_shipping,cargo_flow_change_port_canal_disruption,2026-04-15T21:28:33


## 9. 실행 후 DB 상태 확인

API 실행 후 `tmm_mieventm` 테이블에 row가 증가했는지 확인합니다.


In [11]:
try:
    after_count = get_event_count()
    print("실행 전 건수:", before_count)
    print("실행 후 건수:", after_count)
    if before_count is not None:
        print("증가 건수:", after_count - before_count)
except Exception as exc:
    after_count = None
    print("DB 사후 확인을 건너뜁니다:", exc)


실행 전 건수: 11
실행 후 건수: 17
증가 건수: 6


## 10. DB에 적재된 최신 event 상세 확인

실제 `tmm_mieventm` 테이블에 저장된 최신 event를 직접 조회합니다.


In [12]:
def fetch_latest_events(limit: int = 10) -> pd.DataFrame:
    engine = create_db_engine()
    sql = text("""
        select
            mi_event_id,
            first_created,
            event_title_ko,
            one_line_summary,
            detail_analysis,
            market_and_institution_view,
            occurred_at,
            language,
            dedup_state,
            level_1_code,
            level_2_code,
            taxonomy_version
        from tmm_mieventm
        where del_yn = 'N'
        order by first_created desc, mi_event_id desc
        limit :limit
    """)
    with engine.connect() as conn:
        rows = conn.execute(sql, {"limit": limit}).mappings().all()
    return pd.DataFrame(rows)

try:
    latest_events_df = fetch_latest_events(limit=10)
    latest_events_df
except Exception as exc:
    print("DB 최신 event 조회를 건너뜁니다:", exc)


## 11. 실행 결과 요약

이 데모에서 확인할 수 있는 핵심은 다음과 같습니다.

- `event_flow_agent`는 `/invoke` API를 통해 end-to-end로 실행된다.
- 원천 record를 조회하고, LLM을 통해 record 분석 및 event 추출을 수행한다.
- 추출된 event는 `tmm_mieventm` 테이블에 저장된다.
- 저장된 event를 다시 조회하여 최종 요약을 생성한다.
- 현재 구조는 PoC 및 후속 기능 확장을 위해 단계별 결과를 확인할 수 있도록 구성되어 있다.
